# Lecture 6.4 — Lifecycle hooks: RunHooks and AgentHooks for logging

**Section 06 — Tracing, Observability & Capstone**

In the last three lectures you learned to *read* what an agent run did. Tracing records
the span tree, and the dashboard lets you inspect it after the fact.

This lecture is about the other half of observability. Lifecycle hooks let you run **your
own Python code** at each event in the run, while the run is happening. Same lifecycle,
but now you are in the loop instead of reading the transcript afterwards.

By the end of this notebook you will be able to:

| Goal | How |
|---|---|
| Log every event in a whole workflow | Subclass `RunHooks`, pass it to `Runner.run(hooks=...)` |
| Add side effects for one specific agent | Subclass `AgentHooks`, attach it via `Agent(hooks=...)` |
| Avoid the method-name trap that silently breaks hooks | Know which scope uses which names |
| Build a **per-agent token usage tracker** | The snapshot-and-delta pattern on agent start/end |
| Know which tool calls hooks can and cannot see | Local tools fire hooks, hosted tools do not |

## Cell 1: Install the OpenAI Agents SDK

This notebook needs the `openai-agents` package, which is the Python SDK we use throughout
the course. The install is pinned to a specific version so that the code and the printed
output in this lecture stay reproducible.

If the package is already present in this Colab session, the install simply confirms it and
moves on. The `-q` flag keeps the output quiet.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.19.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.5/928.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.6 MB/s eta 0:00:00


## Cell 2: Configure your OpenAI API key

The SDK reads your key from the `OPENAI_API_KEY` environment variable. In Colab, the safe
place to store it is **Colab Secrets**, not a variable in a cell.

To add the secret:

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your key into the **Value** field.
5. Turn on the **Notebook access** toggle for this notebook.

The cell below reads that secret and writes it into `os.environ` so the SDK can pick it up.

> **Running locally instead of Colab?** Skip the Colab Secrets step and set the variable in
> your terminal before launching Jupyter: `export OPENAI_API_KEY="sk-..."`.

In [2]:
import os

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## Cell 3: Choose the model

We declare the model name once, in a variable, and reuse it in every agent we build. If you
want to run this notebook against a different model, change it here and the whole notebook
follows. Nothing below hardcodes a model string.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

print("Using model:", MODEL_NAME)

Using model: gpt-5.4-mini


## Cell 4: Imports

Most of these are familiar. Here is what is new in this lecture:

| Import | Why we need it |
|---|---|
| `RunHooks` | Base class for run-scoped hooks, passed to `Runner.run(hooks=...)` |
| `AgentHooks` | Base class for agent-scoped hooks, attached via `Agent(hooks=...)` |
| `AgentHookContext` | The context object passed to agent start and end hooks |
| `Tool` | The type of the `tool` argument in the tool hooks |
| `ModelResponse` | The type of the `response` argument in `on_llm_end` |
| `TResponseInputItem` | The item type in the `input_items` list in `on_llm_start` |

Two import paths are worth noting because they are easy to get wrong. `ModelResponse` and
`TResponseInputItem` come from `agents.items`, not from `agents`. And `ToolContext` comes
from `agents.tool_context`.

`Usage` and `ToolContext` are returning guests. You met `Usage` in Lecture 4.7 when we
tracked token cost, and `ToolContext` in Lecture 3.5. Both show up again inside hooks.

In [4]:
from typing import Any

from openai.types.shared import Reasoning

from agents import (
    Agent,
    AgentHookContext,
    AgentHooks,
    ModelSettings,
    RunContextWrapper,
    RunHooks,
    Runner,
    Tool,
    Usage,
    WebSearchTool,
    function_tool,
)
from agents.items import ModelResponse, TResponseInputItem
from agents.tool_context import ToolContext

# Reused for every agent in this notebook.
FAST_SETTINGS = ModelSettings(
    reasoning=Reasoning(effort="none"),
    verbosity="low",
)

print("Imports ready.")

Imports ready.


## Cell 5: The two hook scopes

Before we write any code, here is the mental model.

There are two hook scopes:

- **`RunHooks`** observe the entire `Runner.run(...)` invocation, including handoffs to
  other agents.
- **`AgentHooks`** are attached to a specific agent instance via `agent.hooks`.

The guidance from the SDK docs is short and worth memorising: use `RunHooks` when you want a
single observer for the whole workflow, and `AgentHooks` when one agent needs custom side
effects.

### The context argument changes depending on the event

This catches people out, so note it now:

- Agent start and end hooks receive **`AgentHookContext`**, which wraps your original context
  and carries the shared run usage state.
- LLM, tool, and handoff hooks receive **`RunContextWrapper`**.

For function-tool calls specifically, the context passed to the tool hooks is typically a
`ToolContext`, which exposes `tool_call_id`, `tool_name`, and `tool_arguments`.

### How this relates to tracing

Lectures 6.1 to 6.3 were about tracing. Tracing **records** what happened so you can read it
later. Hooks let you **run your own code** at the moment each event fires. The event sequence
is the same one that produced the span tree you looked at in the dashboard. You are just
attaching to it from inside your own process.

### One rule that applies to every hook

Every hook method is `async def`. Every override you write must be `async def` too.

## Cell 6: RunHooks, the minimal version

Start small. You subclass `RunHooks` and override only the methods you care about. Anything
you do not override stays a no-op, because the base class methods are literally just `pass`.

Below we override three of the seven callbacks:

| Override | Fires |
|---|---|
| `on_agent_start` | Before an agent is invoked, and again each time the current agent changes |
| `on_llm_end` | Immediately after each model call returns |
| `on_agent_end` | When an agent produces its final output |

Note the `hooks=` keyword argument on `Runner.run()`. That is how run-scoped hooks get
wired in. Also note `context.usage` inside `on_agent_end`. That is the same `Usage` object
from Lecture 4.7, and it is available in every hook.

In [5]:
class LoggingHooks(RunHooks):
    async def on_agent_start(self, context, agent):
        print(f"Starting {agent.name}")

    async def on_llm_end(self, context, agent, response):
        print(f"{agent.name} produced {len(response.output)} output items")

    async def on_agent_end(self, context, agent, output):
        print(f"{agent.name} finished with usage: {context.usage}")


assistant = Agent(
    name="Assistant",
    instructions="Be concise.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
)

result = await Runner.run(
    assistant,
    "Explain quines in two sentences.",
    hooks=LoggingHooks(),
)

print("\n--- Final output ---")
print(result.final_output)

Starting Assistant
Assistant produced 1 output items
Assistant finished with usage: Usage(requests=1, input_tokens=20, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=38, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=58, request_usage_entries=[RequestUsage(input_tokens=20, output_tokens=38, total_tokens=58, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=0))])

--- Final output ---
A quine is a program that outputs its own source code. It’s a classic programming trick showing how a program can reproduce its own text without reading its file directly.


## Cell 7: All seven RunHooks callbacks, with an event counter

Now the full surface. `RunHooks` gives you seven callbacks:

| Callback | Signature after `self` | When it fires |
|---|---|---|
| `on_agent_start` | `context, agent` | Before an agent runs, and on each agent change |
| `on_llm_start` | `context, agent, system_prompt, input_items` | Just before each model call |
| `on_llm_end` | `context, agent, response` | Right after each model call returns |
| `on_tool_start` | `context, agent, tool` | Just before a **local** tool runs |
| `on_tool_end` | `context, agent, tool, result` | Right after a **local** tool returns |
| `on_handoff` | `context, from_agent, to_agent` | When control moves between agents |
| `on_agent_end` | `context, agent, output` | When an agent produces its final output |

The `event_counter` is the point of this cell. It numbers the events as they fire, so the
ordering of the lifecycle becomes visible instead of theoretical.

Watch for the interleaving in the output. You should see the agent start, then an LLM call,
then the tool call, then a second LLM call so the model can use the tool result, then the
agent end. That is the same sequence that produced the span tree back in Lecture 6.2, now
showing up as plain Python callbacks.

In [6]:
@function_tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order.

    Args:
        order_id: The order reference, for example A-1234.
    """
    return f"Order {order_id} shipped on 12 March and arrives on 15 March."


class DetailedRunHooks(RunHooks):
    def __init__(self):
        self.event_counter = 0

    def _usage_str(self, usage: Usage) -> str:
        return f"{usage.requests} req, {usage.total_tokens} tokens"

    async def on_agent_start(
        self, context: AgentHookContext, agent: Agent
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] AGENT START: {agent.name}"
            f" | {self._usage_str(context.usage)}"
        )

    async def on_llm_start(
        self,
        context: RunContextWrapper,
        agent: Agent,
        system_prompt: str | None,
        input_items: list[TResponseInputItem],
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] LLM START: {agent.name}"
            f" | {len(input_items)} input items"
        )

    async def on_llm_end(
        self,
        context: RunContextWrapper,
        agent: Agent,
        response: ModelResponse,
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] LLM END: {agent.name}"
            f" | {self._usage_str(context.usage)}"
        )

    async def on_tool_start(
        self,
        context: RunContextWrapper,
        agent: Agent,
        tool: Tool,
    ) -> None:
        self.event_counter += 1
        print(f"[{self.event_counter}] TOOL START: {tool.name}")

    async def on_tool_end(
        self,
        context: RunContextWrapper,
        agent: Agent,
        tool: Tool,
        result: object,
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] TOOL END: {tool.name}"
            f" | result: {str(result)[:50]}"
        )

    async def on_handoff(
        self,
        context: RunContextWrapper,
        from_agent: Agent,
        to_agent: Agent,
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] HANDOFF: "
            f"{from_agent.name} -> {to_agent.name}"
        )

    async def on_agent_end(
        self,
        context: AgentHookContext,
        agent: Agent,
        output: Any,
    ) -> None:
        self.event_counter += 1
        print(
            f"[{self.event_counter}] AGENT END: {agent.name}"
            f" | {self._usage_str(context.usage)}"
        )


support_agent = Agent(
    name="Support Agent",
    instructions="Answer order questions. Use the get_order_status tool.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    tools=[get_order_status],
)

result = await Runner.run(
    support_agent,
    "Where is order A-1234?",
    hooks=DetailedRunHooks(),
)

print("\n--- Final output ---")
print(result.final_output)

[1] AGENT START: Support Agent | 0 req, 0 tokens
[2] LLM START: Support Agent | 1 input items
[3] LLM END: Support Agent | 1 req, 115 tokens
[4] TOOL START: get_order_status
[5] TOOL END: get_order_status | result: Order A-1234 shipped on 12 March and arrives on 15
[6] LLM START: Support Agent | 3 input items
[7] LLM END: Support Agent | 2 req, 283 tokens
[8] AGENT END: Support Agent | 2 req, 283 tokens

--- Final output ---
Order A-1234 shipped on 12 March and is due to arrive on 15 March.


## Cell 8: AgentHooks, and the method names that are not the same

You just watched `RunHooks` observe the whole run. Now let's attach hooks to one specific
agent instead.

**Read this table before you write a single `AgentHooks` subclass.** It is the most important
thing in the lecture:

| Event | `RunHooks` method | `AgentHooks` method |
|---|---|---|
| Agent begins | `on_agent_start` | **`on_start`** |
| Agent finishes | `on_agent_end` | **`on_end`** |
| Before LLM call | `on_llm_start` | `on_llm_start` |
| After LLM call | `on_llm_end` | `on_llm_end` |
| Tool invoked | `on_tool_start` | `on_tool_start` |
| Tool returned | `on_tool_end` | `on_tool_end` |
| Handoff | `on_handoff` | `on_handoff` |

Five of the seven names match. The two agent lifecycle ones do not. `RunHooks` says
`on_agent_start` and `on_agent_end`. `AgentHooks` says `on_start` and `on_end`.

The cell below defines a `SpecialistHooks` class with `on_start`, `on_end`, and `on_tool_start`.
It then builds an agent and passes the hooks through the `hooks=` field on the `Agent`
constructor. You can also assign them later with `agent.hooks = SpecialistHooks()`.

Inside `on_tool_start` we check whether the context is a `ToolContext`. For function tools it
usually is, which gives us `tool_call_id`, `tool_name`, and `tool_arguments`. That is the same
`ToolContext` you met in Lecture 3.5.

In [7]:
class SpecialistHooks(AgentHooks):
    # NOTE: on_start, NOT on_agent_start.
    # AgentHooks uses different method names to RunHooks.
    async def on_start(
        self, context: AgentHookContext, agent: Agent
    ) -> None:
        print(f">>> [AgentHooks] {agent.name} started")

    async def on_end(
        self,
        context: AgentHookContext,
        agent: Agent,
        output: Any,
    ) -> None:
        print(f">>> [AgentHooks] {agent.name} finished")

    async def on_tool_start(
        self,
        context: RunContextWrapper,
        agent: Agent,
        tool: Tool,
    ) -> None:
        # For function tools, context is typically a ToolContext.
        if isinstance(context, ToolContext):
            print(
                f">>> [AgentHooks] tool {tool.name} "
                f"call_id={context.tool_call_id}"
            )
        else:
            print(f">>> [AgentHooks] tool {tool.name}")


specialist = Agent(
    name="Specialist",
    instructions="You are a specialist. Use the tool, then answer briefly.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    tools=[get_order_status],
    hooks=SpecialistHooks(),
)

result = await Runner.run(specialist, "Check on order B-9876 for me.")

print("\n--- Final output ---")
print(result.final_output)

>>> [AgentHooks] Specialist started
>>> [AgentHooks] tool get_order_status call_id=call_pcwzNYnPXL5wPj2XGRRziP5k
>>> [AgentHooks] Specialist finished

--- Final output ---
Order B-9876 shipped on 12 March and arrives on 15 March.


## Cell 9: The silent failure

This cell exists to show you a bug rather than a feature.

`BrokenHooks` below subclasses `AgentHooks` but defines `on_agent_start`, which is a
`RunHooks` method name. Python is perfectly happy with this. You have simply added an extra
method to a subclass. Nothing in the SDK ever calls it, and the base class `on_start` it
*would* have called is just `pass`.

So there is no error. No warning. No traceback. The only symptom is silence where your
logging should have been.

Run it and look carefully at what is **missing** from the output.

In [8]:
class BrokenHooks(AgentHooks):
    # WRONG METHOD NAME for AgentHooks. This will never fire.
    async def on_agent_start(self, context, agent) -> None:
        print("You will NEVER see this line")


broken_agent = Agent(
    name="Broken Hooks Agent",
    instructions="Be concise.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    hooks=BrokenHooks(),
)

result = await Runner.run(broken_agent, "Say hello in five words.")

print("Output:", result.final_output)
print(
    "\nNotice: no hook output above. on_agent_start is a RunHooks method "
    "name. On AgentHooks the correct name is on_start. Python accepted the "
    "method silently because it is just an extra attribute on the subclass."
)

Output: Hello there, how are you?

Notice: no hook output above. on_agent_start is a RunHooks method name. On AgentHooks the correct name is on_start. Python accepted the method silently because it is just an extra attribute on the subclass.


## Cell 10: The loud failure, for contrast

There is a related mistake that behaves in the opposite way, and the contrast is worth seeing
back to back.

In Cell 9 you used the wrong **method name** and got silence. Here we use the wrong **type**.
We build a perfectly valid `AgentHooks` subclass and then pass it to `Runner.run(hooks=...)`,
which expects run-scoped hooks.

The SDK validates this at the start of the run and raises a `TypeError` with a message that
tells you exactly what to do instead. We catch it so the notebook keeps running.

So the two mistakes fail in opposite directions. Wrong type, immediate error. Wrong method
name, nothing at all. The loud one is the easy one. Silence is what costs you an afternoon.

In [9]:
class MisplacedHooks(AgentHooks):
    async def on_start(self, context, agent) -> None:
        print("This never gets a chance to run.")


try:
    result = await Runner.run(
        broken_agent,
        "Say hello.",
        hooks=MisplacedHooks(),  # AgentHooks passed where RunHooks is expected
    )
except TypeError as e:
    print("TypeError raised by the SDK:")
    print(" ", e)

TypeError raised by the SDK:
  Run hooks must be instances of RunHooks. Received agent-scoped hooks (MisplacedHooks). Attach AgentHooks to an Agent via Agent(..., hooks=...).


## Cell 11: Both scopes together, across a handoff

Now we put the two scopes side by side in one run and watch them interleave.

The setup: a triage agent hands off to a billing agent. `RunHooks` is attached at the runner
level. Both agents also carry their own `AgentHooks`. Every hook prints a label saying which
scope it came from, so the output reads as a trace of who saw what.

### About `on_handoff`, and a discrepancy worth knowing

Both scopes have a method called `on_handoff`, and the parameter lists are different:

| Scope | Signature after `self` |
|---|---|
| `RunHooks` | `context, from_agent, to_agent` |
| `AgentHooks` | `context, agent, source` |

Now the part that is easy to get wrong. The docstring in `agents/lifecycle.py` describes
`AgentHooks.on_handoff` as firing on the agent **being handed off to**. The actual call site
in the run loop does something different. It fires the hooks belonging to the agent that is
**handing off**, and passes `agent` as the destination and `source` as the sender.

In other words, in the run you are about to execute, the triage agent's `on_handoff` fires and
the billing agent's does not. The documentation and the implementation disagree here, so trust
what you see in the output below rather than the docstring.

This is a good habit in general. When a hook does not fire and you cannot see why, the call
site in the SDK source is the authority.

In [11]:
class HandoffWatchRunHooks(RunHooks):
    async def on_agent_start(self, context, agent) -> None:
        print(f"[RunHooks]    agent start : {agent.name}")

    async def on_handoff(self, context, from_agent, to_agent) -> None:
        print(
            f"[RunHooks]    handoff     : from_agent={from_agent.name}, "
            f"to_agent={to_agent.name}"
        )


class TriageAgentHooks(AgentHooks):
    async def on_start(self, context, agent) -> None:
        print(f"[AgentHooks:Triage]  on_start fired for {agent.name}")

    async def on_handoff(self, context, agent, source) -> None:
        print(
            f"[AgentHooks:Triage]  on_handoff fired. "
            f"agent={agent.name}, source={source.name}"
        )


class BillingAgentHooks(AgentHooks):
    async def on_start(self, context, agent) -> None:
        print(f"[AgentHooks:Billing] on_start fired for {agent.name}")

    async def on_handoff(self, context, agent, source) -> None:
        print(
            f"[AgentHooks:Billing] on_handoff fired. "
            f"agent={agent.name}, source={source.name}"
        )


billing_agent = Agent(
    name="Billing Agent",
    instructions="Handle billing questions. Answer in one short sentence.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    hooks=BillingAgentHooks(),
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You route customer messages. "
        "Hand off anything about charges, refunds or invoices to the Billing Agent."
    ),
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    handoffs=[billing_agent],
    hooks=TriageAgentHooks(),
)

result = await Runner.run(
    triage_agent,
    "I was charged twice for my subscription this month.",
    hooks=HandoffWatchRunHooks(),
)

print("\n--- Final output ---")
print(result.final_output)

[RunHooks]    agent start : Triage Agent
[AgentHooks:Triage]  on_start fired for Triage Agent
[RunHooks]    handoff     : from_agent=Triage Agent, to_agent=Billing Agent
[AgentHooks:Triage]  on_handoff fired. agent=Billing Agent, source=Triage Agent
[RunHooks]    agent start : Billing Agent
[AgentHooks:Billing] on_start fired for Billing Agent

--- Final output ---
I’ve escalated this to billing support so they can review the duplicate charge and help resolve it.


## Cell 12: Reading `turn_input` from `AgentHookContext`

Agent start and end hooks receive an `AgentHookContext`. It wraps your original context and
carries the shared run usage state, and it also exposes `turn_input`, which is the list of
input items the agent received for this turn.

This cell defines a hooks class that prints `turn_input` on start, then attaches it to an
agent and runs it.

Two limits worth remembering. `turn_input` is available on the agent start and end hooks only.
The LLM, tool, and handoff hooks receive a plain `RunContextWrapper` instead, which does not
carry it.

This turns out to be a practical debugging tool. When you use the handoff input filters from
Lecture 5.3 and the receiving agent behaves strangely, printing `turn_input` on that agent's
`on_start` shows you exactly what it was handed after filtering.

In [12]:
class TurnInputHooks(AgentHooks):
    async def on_start(
        self, context: AgentHookContext, agent: Agent
    ) -> None:
        print(f"{agent.name} received turn_input:")
        for item in context.turn_input:
            print(f"  {item}")


inspector = Agent(
    name="Inspector",
    instructions="Answer in one short sentence.",
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    hooks=TurnInputHooks(),
)

result = await Runner.run(inspector, "What is a lifecycle hook?")

print("\n--- Final output ---")
print(result.final_output)

Inspector received turn_input:
  {'content': 'What is a lifecycle hook?', 'role': 'user'}

--- Final output ---
A lifecycle hook is a function or callback that runs at a specific stage in an object’s or component’s life, such as creation, update, or destruction.


## Cell 13: Per-agent token usage, the Lecture 4.7 payoff

Back in Lecture 4.7 you learned that `result.context_wrapper.usage` is aggregated across
**all** agents in a run, with no built-in per-agent breakdown. We said at the time that
building one requires capturing usage snapshots in hooks and computing the delta. This is
that cell.

The pattern is three lines of idea:

1. `context.usage` is the **shared, cumulative** `Usage` object. It is the same object in
   every hook, and it keeps climbing through the whole run.
2. On `on_agent_start`, record the running total for that agent.
3. On `on_agent_end`, subtract the recorded total from the current total. The difference is
   what that agent consumed.

We accumulate into a dictionary rather than overwriting, because an agent can be entered more
than once in a run.

**One honest caveat.** This attributes tokens to whichever agent was active when they were
consumed. That is an approximation rather than a guarantee, and it gets fuzzier if you have
concurrent work in flight. For the everyday question of which agent is costing you the most,
it is more than good enough.

In [13]:
class PerAgentUsageHooks(RunHooks):
    def __init__(self):
        self.snapshots: dict[str, int] = {}
        self.per_agent_tokens: dict[str, int] = {}

    async def on_agent_start(
        self, context: AgentHookContext, agent: Agent
    ) -> None:
        # Snapshot cumulative tokens at the moment this agent starts.
        self.snapshots[agent.name] = context.usage.total_tokens

    async def on_agent_end(
        self,
        context: AgentHookContext,
        agent: Agent,
        output: Any,
    ) -> None:
        start_tokens = self.snapshots.get(agent.name, 0)
        delta = context.usage.total_tokens - start_tokens
        self.per_agent_tokens[agent.name] = (
            self.per_agent_tokens.get(agent.name, 0) + delta
        )


usage_hooks = PerAgentUsageHooks()

result = await Runner.run(
    triage_agent,
    "My invoice from last month looks wrong, can you check it?",
    hooks=usage_hooks,
)

print("Per-agent token usage:")
for name, tokens in usage_hooks.per_agent_tokens.items():
    print(f"  {name}: {tokens} tokens")

print(
    f"\nRun total (from Lecture 4.7): "
    f"{result.context_wrapper.usage.total_tokens} tokens"
)

[AgentHooks:Triage]  on_start fired for Triage Agent
[AgentHooks:Triage]  on_handoff fired. agent=Billing Agent, source=Triage Agent
[AgentHooks:Billing] on_start fired for Billing Agent
Per-agent token usage:
  Billing Agent: 87 tokens

Run total (from Lecture 4.7): 185 tokens


## Cell 14: Hosted tools do not fire tool hooks

One boundary left, and it is the one people trip over when they wire up an audit log and find
gaps in it.

The `on_tool_start` and `on_tool_end` hooks apply only to **local** tools. They do not include
hosted tools that run on the OpenAI server side, such as `WebSearchTool`, `FileSearchTool`,
`CodeInterpreterTool`, `HostedMCPTool`, or other built-in hosted tools.

The reason is mechanical rather than arbitrary. A hosted tool executes on OpenAI's servers.
Your Python process never sees the invocation, so there is nothing local for a hook to fire on.
This confirms what we said back in Lecture 3.3.

The cell below builds one agent with both kinds of tool registered, a hosted `WebSearchTool`
and a local `local_lookup` function tool. Then it runs a query that steers the model to the
local tool, so you can see the hook fire. If the model had reached for web search instead, the
hook output would simply be absent.

If you need visibility into hosted tool calls, tracing is the answer, not hooks. That is
exactly the split between Lectures 6.1 to 6.3 and this one.

In [14]:
@function_tool
def local_lookup(query: str) -> str:
    """A local function tool that DOES fire hooks.

    Args:
        query: The search query.
    """
    return f"Local result for: {query}"


class ToolWatchHooks(RunHooks):
    async def on_tool_start(self, context, agent, tool: Tool) -> None:
        print(f"  HOOK FIRED for tool: {tool.name}")

    async def on_tool_end(self, context, agent, tool: Tool, result: object) -> None:
        print(f"  HOOK FIRED for tool end: {tool.name}")


mixed_agent = Agent(
    name="Mixed Tools Agent",
    instructions=(
        "Use web search for current events. "
        "Use local_lookup for internal company queries."
    ),
    model=MODEL_NAME,
    model_settings=FAST_SETTINGS,
    tools=[WebSearchTool(), local_lookup],
)

print("Running with a query that uses the LOCAL tool:")
result = await Runner.run(
    mixed_agent,
    "Do a local_lookup for 'employee handbook'.",
    hooks=ToolWatchHooks(),
)

print("\nOutput:", result.final_output[:200])
print(
    "\nNote: if the agent had used WebSearchTool instead, "
    "NO hook would have fired for it."
)

Running with a query that uses the LOCAL tool:
  HOOK FIRED for tool: local_lookup
  HOOK FIRED for tool end: local_lookup

Output: Local lookup done for **employee handbook**.

Note: if the agent had used WebSearchTool instead, NO hook would have fired for it.


## Cell 15: RunHooks vs AgentHooks, one page to keep

Everything from this notebook, in one place.

### Method names

| Event | `RunHooks` | `AgentHooks` |
|---|---|---|
| Agent begins | `on_agent_start` | `on_start` |
| Agent finishes | `on_agent_end` | `on_end` |
| Before LLM call | `on_llm_start` | `on_llm_start` |
| After LLM call | `on_llm_end` | `on_llm_end` |
| Tool invoked | `on_tool_start` | `on_tool_start` |
| Tool returned | `on_tool_end` | `on_tool_end` |
| Handoff | `on_handoff(context, from_agent, to_agent)` | `on_handoff(context, agent, source)` |

### Which one do I reach for

| Goal | Use |
|---|---|
| One observer for the whole workflow | `RunHooks` |
| Side effects for one specific agent | `AgentHooks` |
| Per-agent usage or cost tracking | `RunHooks` with the snapshot-and-delta pattern |
| Audit log of every local tool call in a run | `RunHooks` |
| Custom behaviour only when one specialist runs | `AgentHooks` |
| Instrument hosted tool calls | Neither. Use tracing, from Lectures 6.1 to 6.3 |

### Things that will save you time later

- Every hook method is `async def`.
- Anything you do not override is a silent no-op.
- Agent start and end hooks get `AgentHookContext`, which carries `turn_input` and `usage`.
  LLM, tool, and handoff hooks get `RunContextWrapper`.
- For function tools, the tool hook context is typically a `ToolContext` with `tool_call_id`,
  `tool_name`, and `tool_arguments`.
- Passing `AgentHooks` to `Runner.run(hooks=...)` raises a `TypeError`. Using a `RunHooks`
  method name on an `AgentHooks` subclass raises nothing at all.
- `AgentHooks.on_handoff` fires on the agent handing off, not the one receiving, whatever the
  docstring says.

### Not covered here

Nested `as_tool()` runs take their own `hooks` parameter, which we are not going into.
Realtime agent hooks, tool guardrails, and custom trace processors all come later in the
course.